[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-1/router.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239412-lesson-5-router)

# Router

## 回顾

我们构建了一个使用 `messages` 作为状态并包含带有绑定工具的聊天模型的图。

我们看到该图可以：

* 返回工具调用
* 返回自然语言响应

## 目标

我们可以将其视为一个 Router（路由器），其中聊天模型根据用户输入在直接响应和工具调用之间进行路由。

这是一个简单的 Agent 示例，LLM 通过调用工具或直接响应来指导控制流。

![Screenshot 2024-08-21 at 9.24.09 AM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbac6543c3d4df239a4ed1_router1.png)

让我们扩展图，使其能够处理两种输出！

为此，我们可以使用两个思路：

(1) 添加一个用于调用工具的节点。

(2) 添加一个条件边，它会查看聊天模型的输出，并路由到工具调用节点，或者在没有工具调用时直接结束。



In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_deepseek langchain_core langgraph langgraph-prebuilt

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("DEEPSEEK_API_KEY")

In [ ]:
from langchain_deepseek import ChatDeepSeek

def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b

llm = ChatDeepSeek(model="deepseek-chat")
llm_with_tools = llm.bind_tools([multiply])

我们使用[内置的 `ToolNode`](https://langchain-ai.github.io/langgraph/reference/agents/#langgraph.prebuilt.tool_node.ToolNode)，只需传入工具列表即可初始化。

我们使用[内置的 `tools_condition`](https://langchain-ai.github.io/langgraph/reference/agents/#langgraph.prebuilt.tool_node.tools_condition) 作为条件边。

In [ ]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

# 节点
def tool_calling_llm(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

# 构建图
builder = StateGraph(MessagesState)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode([multiply]))
builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    # 如果 assistant 的最新消息（结果）是工具调用 -> tools_condition 路由到 tools
    # 如果 assistant 的最新消息（结果）不是工具调用 -> tools_condition 路由到 END
    tools_condition,
)
builder.add_edge("tools", END)
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
from langchain_core.messages import HumanMessage
messages = [HumanMessage(content="Hello, what is 2 multiplied by 2?")]
messages = graph.invoke({"messages": messages})
for m in messages['messages']:
    m.pretty_print()

现在，我们可以看到图运行了工具！

它响应了一个 `ToolMessage`。

## Studio

**⚠️ 注意**

自录制这些视频以来，我们已更新了 Studio，使其现在可以在本地运行并通过浏览器访问。这是运行 Studio 的首选方式，而不是使用视频中展示的桌面应用。它现在被称为 *LangSmith Studio* 而不是 *LangGraph Studio*。详细的设置说明可在课程开头的"Getting Setup"指南中找到。你可以[在此](https://docs.langchain.com/langsmith/studio)查看 Studio 的说明，以及[在此](https://docs.langchain.com/langsmith/quick-start-studio#local-development-server)查看本地部署的具体细节。

要在本模块的 `/studio` 目录中启动本地开发服务器，请在终端中运行以下命令：

```
langgraph dev
```

你应该看到以下输出：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

打开浏览器并导航到上面显示的 Studio UI。

在 Studio 中加载 `router`，它使用 `module-1/studio/langgraph.json` 中设置的 `module-1/studio/router.py`。

In [ ]:
if 'google.colab' in str(get_ipython()):
    raise Exception("Unfortunately LangGraph Studio is currently not supported on Google Colab")